# 27. Weight change and input sensitivity, measured together

### Why both numbers

Two different questions get asked about a model after unlearning.

**How far did the weights move?** The adapter plasticity study in
`reference/adapter-plasticity-study` answers this with one number it calls `rel_dW`:

    d_W = || W_eff - W0 || / || W0 ||

W0 is the weights before, W_eff is the weights now, and the bars mean the square root of
the sum of squared entries. If d_W is 0.05, the weights moved by 5 percent of their own
size.

**Does the model still work?** This study answers that with input sensitivity: how far any
single output score travels across the test set. A working model moves a lot, because the
picture decides the scores. A model that has stopped reading its input barely moves.

### Why they must both be read

This notebook exists because the two disagree, and the disagreement is the study's main
finding.

Over 100 unlearning steps the generated weights lose about 43 percent of their size. That
is an ordinary amount of movement. Over the same steps input sensitivity falls by a factor
of several million. A weight measurement alone would have shown nothing alarming at the
exact point where the model stopped working.

### What this notebook does

It applies one unlearning request to a saved model and records both numbers every 20
steps, on the task being removed and on the tasks being kept.

Nothing here is new science. Notebook 21 measured this already. What is new is that both
numbers now come from one shared module, `uncle/measure.py`, instead of from a function
copied into five notebooks in five versions that had drifted apart. Running this notebook
confirms the shared module reproduces what the separate copies produced.

### Cost

One unlearning request on a saved source model. A few minutes on A100, slower on CPU. It
trains no source model, writes nothing into the study, and cannot disturb a resume.

## 1. Load the code

This notebook runs from `main`, not from the study's pinned revision `65e2e7c8`.

That is deliberate and it is safe here. `uncle/measure.py` was written after the pin, so a
checkout at the pinned revision does not contain it. The pin exists to keep *training*
reproducible, and this notebook trains no source model: it loads saved weights, applies one
unlearning request with the study's own code, and measures. The unlearning loop itself is
checked against the pinned revision in section 5 before it runs.

In [ ]:
from pathlib import Path
import sys, os, json, subprocess, importlib.util
IN_COLAB = "google.colab" in sys.modules
# The revision the study's results were produced at. Used as a check, not a checkout.
STUDY_CODE_VERSION = "65e2e7c850fd6f1852a99d9e81780c27f799c926"
if any(name == "uncle" or name.startswith("uncle.") for name in sys.modules):
    raise RuntimeError("Restart the runtime before loading the code.")
if IN_COLAB:
    REPO = Path("/content/path-measure")
    if not REPO.exists():
        subprocess.run(["git", "clone", "https://github.com/sumitasthana/PATH.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "uncle/task_partition.json").exists()), None)
    if REPO is None:
        raise FileNotFoundError("Open this notebook from a PATH checkout.")
for package in ("boto3", "tqdm", "matplotlib"):
    if importlib.util.find_spec(package) is None:
        if not IN_COLAB:
            raise ImportError("Install the storage and extras dependencies first.")
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", package], check=True)
sys.path.insert(0, str(REPO))

from uncle.storage import R2Store
from uncle.paired_study import StudyStore, jobs
from uncle import checkpoint
from uncle.config import Config, random_stream
from uncle.hypernet import build_target, HyperNetwork
from uncle.trainer import UnCLe
from uncle.streams import build_tasks
from uncle.study_training import restore_rng
# The shared measurements. Both numbers come from here, and nowhere else.
from uncle.measure import probe, relative_change, weight_size

# The progress panel draws bars only, so it cannot change a result. It lives in
# scripts/ rather than in the pinned implementation. Fall back to plain bars if
# it cannot be imported, rather than failing the run over a display helper.
sys.path.insert(0, str(REPO / 'scripts'))
try:
    from dashboard import Dashboard
except ImportError as problem:
    print('Progress panel unavailable, using plain bars:', problem)

    class Dashboard:
        """Just enough of the panel to keep the notebook running."""

        def __init__(self, *args, **kwargs): pass
        def __enter__(self): return self
        def __exit__(self, *args): return False
        def note(self, text): print(text, flush=True)

        def bar(self, total, description, unit='step'):
            from uncle.telemetry import progress_bar
            return progress_bar(total, description)

import torch
HEAD = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Running at:", HEAD)
print("Study pinned revision:", STUDY_CODE_VERSION)
print("PyTorch:", torch.__version__, "CUDA build:", torch.version.cuda)

## 2. Settings

Everything matches the saved study. `INTERVAL` only decides how often a measurement is
taken; it does not change the unlearning.

In [ ]:
STUDY_ID = "paired_generalization_v2"
BUCKET = "uncle-experiments"

ORDER = "order_01"
SEED = 0                 # 0, 1 or 2
FORGET_TASK = "3"        # "3" or "14"

# Measure every this many unlearning steps. The study's own request is 100 steps,
# so 20 gives six readings including the one before anything happens.
INTERVAL = 20

# Upload the result files to the study's diagnostics folder as well as saving
# them locally. The local copy is written either way.
SAVE_TO_R2 = True

# Disconnect the Colab runtime once the files are written and verified, so a
# finished notebook stops consuming GPU time. It never releases the runtime if
# anything failed to save.
RELEASE_GPU_WHEN_DONE = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = Path("/content/data/tiny-imagenet-200") if IN_COLAB else REPO / "data/tiny-imagenet-200"
print("Device:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Matching the study runner: no fast paths that change the arithmetic.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

## 3. Connect to R2

Supply `R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` through Colab Secrets or
environment variables. Their values are never printed or saved.

In [ ]:
def secret(name):
    value = os.environ.get(name)
    if not value and IN_COLAB:
        from google.colab import userdata
        value = userdata.get(name)
    if not value:
        raise RuntimeError("Missing secret: " + name)
    return value

store = R2Store.connect(secret("R2_ENDPOINT"), BUCKET,
    secret("R2_ACCESS_KEY_ID"), secret("R2_SECRET_ACCESS_KEY"))
store.progress = True
CACHE = REPO / "outputs/paired_study" / STUDY_ID
CACHE.mkdir(parents=True, exist_ok=True)
print("Connected to R2 bucket:", BUCKET)

## 4. Load the saved model

The source model has learned the eight tasks of order 01 and forgotten nothing. It is the
model the study's own unlearning branches start from.

In [ ]:
PREFIX = "uncle/paired_generalization/" + STUDY_ID
plan = json.loads(store.download(PREFIX + "/study.json", CACHE / "cache").read_text())
if plan["code_version"] != STUDY_CODE_VERSION:
    raise ValueError("This study was produced at a different revision than this notebook expects.")

storage = StudyStore(store, plan, CACHE, cleanup_completed=False)
ALL_JOBS = jobs(plan)
ORDER_TASKS = tuple(plan["orders"][ORDER])
CLASSES = plan["config"]["classes_per_task"]
EVAL_BATCH = plan["config"]["eval_batch_size"]

tasks = build_tasks(root=DATA_ROOT, partition_path=REPO / "uncle/task_partition.json",
                    partition_seed=42, include=ORDER_TASKS, download=True)

source_job = next(j for j in ALL_JOBS if j["id"] == f"sources/{ORDER}/seed_{SEED}")
state = storage.load(source_job)
if state is None or state["report"]["status"] != "complete":
    raise RuntimeError(f"Source model for seed {SEED} is not finished in R2.")

# Forgetting protects every learned task except its own target. Copied from the
# study runner so this request matches the one the study makes.
PROTECTED = [t for t in state["seen"] if t != FORGET_TASK]
print("Source:", source_job["id"], "| removing task", FORGET_TASK,
      "| protecting", len(PROTECTED), "tasks")

## 5. Check this code against the pinned revision

The unlearning loop below is the study's own `_forget_generated`, imported rather than
copied. This section checks that running from `main` has not changed what it does.

The check treats two kinds of file differently, because "changed" and "changed in a way
that matters" are not the same thing.

**`trainer.py` and `paired_diagnostic.py` must be byte for byte identical.** They hold the
unlearning arithmetic. Any difference at all stops the run.

**`hypernet.py` is allowed to have lines added, but none removed or altered.** It has one
such addition: a later commit replaced ResNet's `AdaptiveAvgPool2d` with a plain mean,
because the pooling layer has no exact backward pass on a GPU and stops a run with exact
arithmetic enabled. The two compute the same value.

That claim is not taken on trust. The cell below prints every added line so you can see what
you are accepting, then tests the replacement against the layer it replaced, on this
machine and this device. If the two ever disagree, the run stops.

In [ ]:
import hashlib

def pinned_text(name):
    return subprocess.check_output(["git", "-C", str(REPO), "show", f"{STUDY_CODE_VERSION}:{name}"])

def head_text(name):
    return subprocess.check_output(["git", "-C", str(REPO), "show", f"HEAD:{name}"])

# The unlearning arithmetic. Any difference at all stops the run.
for name in ("uncle/paired_diagnostic.py", "uncle/trainer.py"):
    if hashlib.sha256(head_text(name)).hexdigest() != hashlib.sha256(pinned_text(name)).hexdigest():
        raise RuntimeError(f"{name} differs from the pinned revision. Do not trust this run.")
print("Unlearning arithmetic matches the pinned revision exactly.")

# hypernet.py may have lines added, but nothing removed or altered. A removed
# line means existing behaviour changed, which this notebook cannot vouch for.
diff = subprocess.check_output(
    ["git", "-C", str(REPO), "diff", "--unified=0", STUDY_CODE_VERSION, "HEAD",
     "--", "uncle/hypernet.py"], text=True).splitlines()
removed = [l[1:] for l in diff if l.startswith("-") and not l.startswith("---")]
added = [l[1:] for l in diff if l.startswith("+") and not l.startswith("+++")]
if removed:
    raise RuntimeError("uncle/hypernet.py has lines removed or altered since the pin:\n"
                       + "\n".join(removed) + "\nDo not trust this run.")
if added:
    print(f"\nuncle/hypernet.py has {len(added)} added lines and none removed. They are:\n")
    for line in added:
        print("   ", line)
else:
    print("uncle/hypernet.py matches the pinned revision exactly.")

Now test the one addition that changes what the model computes.

`AdaptiveAvgPool2d((1, 1))` averages each feature map down to a single number. So does
`x.mean((2, 3), keepdim=True)`. They should agree exactly, but floating-point addition
depends on the order the numbers are added, and a GPU may choose a different order. So the
comparison runs on the device this notebook is using, at the shapes a ResNet-50 actually
produces, rather than being assumed.

In [ ]:
import torch.nn as nn

# ResNet-50 reaches the pooling layer with 2048 channels. Tiny ImageNet images
# are 64x64 and this study keeps the stem's max-pool, so the map is small.
# Several shapes are tried because the tolerance question is about how many
# numbers get added together.
worst = 0.0
for shape in ((8, 2048, 2, 2), (8, 2048, 4, 4), (4, 512, 8, 8)):
    sample = torch.randn(*shape, device=DEVICE)
    stock = nn.AdaptiveAvgPool2d((1, 1))(sample)
    replacement = sample.mean((2, 3), keepdim=True)
    if stock.shape != replacement.shape:
        raise RuntimeError(f"Shapes differ at {shape}: {stock.shape} against {replacement.shape}")
    worst = max(worst, (stock - replacement).abs().max().item())
    print(f"  {str(shape):<22} max difference {(stock - replacement).abs().max().item():.3e}")

# The two must agree to float32's own precision. Anything larger means the
# replacement is not the equivalent it claims to be.
if worst > 1e-6:
    raise RuntimeError(f"The pooling replacement differs by {worst:.3e}. Do not trust this run.")
print(f"\nThe pooling replacement matches the layer it replaced on {DEVICE}. "
      f"Worst difference {worst:.3e}.")
print("The model this notebook builds computes what the study's model computed.")

## 6. Rebuild the model

In [ ]:
from uncle.paired_diagnostic import _forget_generated

config = Config(**{**plan["config"], "seed": SEED, "device": DEVICE,
                   "requests": tuple(("learn", t) for t in ORDER_TASKS) + (("forget", FORGET_TASK),),
                   "burn_in": 100})
target = build_target(config)
hypernet = HyperNetwork(target, config)
trainer = UnCLe(hypernet, config, target, tasks, progress=True)
checkpoint.restore(state, hypernet=hypernet, uncle=trainer)
restore_rng(state["all_rng"])
print("Model restored. Tasks learned:", list(state["seen"]))

## 7. Capture the reference weights

The relative weight change is measured against this moment, so it must be captured before
anything changes. The clone matters: `weights_for` returns tensors that are regenerated
each call, and without a copy the reference would follow the model as it changes.

The check that follows is the same one the adapter study makes at the start of its runs:
compared against itself, the change must be exactly zero.

In [ ]:
reference = {name: value.detach().clone()
             for name, value in hypernet.weights_for(FORGET_TASK).items()}

# Comparing the reference with itself must give exactly zero. A non-zero value
# here would mean the capture did not take a real copy.
check = relative_change(hypernet.weights_for(FORGET_TASK), reference)
if check != 0.0:
    raise RuntimeError(f"Reference capture is wrong: self-comparison gave {check}")
print(f"Reference captured: {len(reference)} tensors, "
      f"{sum(v.numel() for v in reference.values()):,} numbers. Self-check {check}.")

## 7b. Record which neurons fire, before any unlearning

Accuracy tells us the forgotten task stops working. It does not say what stopped. This
section records, for every ReLU in the generated ResNet50, how strongly each channel
responds and how many channels respond at all.

We take the reading twice: here, before unlearning, and again in section 9b afterwards.
Without the "before" picture a quiet layer proves nothing, because it may always have
been quiet.

We also read a task we are keeping. If the forgotten task goes flat and the kept task
does not, the effect is specific to that task. If both go flat, we damaged the generator
itself.

In [ ]:
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.func import functional_call

# One batch is enough. We want the shape of the response, not a precise average,
# and every extra batch costs GPU time for no extra information.
ACTIVATION_IMAGES = 256

# A channel counts as silent when its strongest response anywhere in the batch
# falls below this fraction of what that same channel did before unlearning.
# The threshold has to be relative: the unlearning objective drives the whole
# network toward zero, so the absolute scale shifts under us.
SILENT_FRACTION = 0.01


def activation_profile(task, images=ACTIVATION_IMAGES):
    """Per-channel response of every ReLU in the generated network, for one task.

    Returns {layer name: {"mean": per-channel mean size, "peak": per-channel
    largest response}}, both as plain lists.

    Evaluation mode is not optional here. A BatchNorm layer in training mode
    divides a nearly dead signal by its own tiny spread, which returns it to
    normal size and hides exactly the effect we are measuring. Section 10 of
    the conclusion report records that trap. The previous modes are restored
    afterwards, so a reading taken mid-run does not disturb anything.
    """
    was_training = hypernet.training, target.training
    hypernet.eval()
    target.eval()

    # The weights this task's code generates, and the task's own BatchNorm
    # statistics. Both are copied, so measuring cannot change the model.
    weights = hypernet.weights_for(task)
    buffers = {name: value.clone() for name, value in trainer.task_buffers[task].items()}

    captured = {}
    handles = []

    def make_hook(name):
        def hook(_module, _inputs, output):
            # ResNet reuses one ReLU module several times inside a block, so
            # this fires more than once for the same name. The call number
            # keeps those apart instead of silently overwriting.
            key = f"{name}#{sum(1 for k in captured if k.startswith(name + '#'))}"
            # Reduce to per-channel numbers inside the hook. Keeping whole
            # feature maps for 50 layers would not fit in memory.
            flat = output.detach().abs()
            dims = (0, 2, 3) if flat.dim() == 4 else (0,)
            captured[key] = {"mean": flat.mean(dim=dims).cpu(),
                             "peak": flat.amax(dim=dims).cpu()}
        return hook

    try:
        for name, module in target.named_modules():
            if isinstance(module, nn.ReLU):
                handles.append(module.register_forward_hook(make_hook(name)))
        loader = DataLoader(images_of(task), batch_size=EVAL_BATCH, shuffle=False)
        seen = 0
        with torch.no_grad():
            for batch, _labels in loader:
                captured.clear()      # keep only the last batch's reading
                functional_call(target, (weights, buffers),
                                (batch.to(DEVICE),), strict=True)
                seen += batch.shape[0]
                if seen >= images:
                    break
    finally:
        for handle in handles:
            handle.remove()
        hypernet.train(was_training[0])
        target.train(was_training[1])

    return {key: {"mean": value["mean"].tolist(), "peak": value["peak"].tolist()}
            for key, value in captured.items()}


# The images each profile runs on. Always the forgotten task's own test images,
# even when we profile a kept task, so the two profiles differ only by which
# code generated the weights and not by what they are looking at.
def images_of(_task):
    return tasks[FORGET_TASK]["test"]


# A kept task to act as the control. The first protected task will do.
CONTROL_TASK = PROTECTED[0]

profiles_before = {
    FORGET_TASK: activation_profile(FORGET_TASK),
    CONTROL_TASK: activation_profile(CONTROL_TASK),
}
layers = list(profiles_before[FORGET_TASK])
channels = sum(len(v["mean"]) for v in profiles_before[FORGET_TASK].values())
print(f"Recorded {len(layers)} ReLU readings, {channels} channels in total.")
print(f"Forgotten task {FORGET_TASK}, control task {CONTROL_TASK}, "
      f"{ACTIVATION_IMAGES} images each.")

## 8. Run the unlearning request, measuring as it goes

One reading before anything happens, then one every `INTERVAL` steps.

Each reading records, for the task being removed: accuracy, how many classes the model ever
chooses, input sensitivity, the relative weight change against section 6, and the raw weight
size. It also records the mean accuracy of the tasks being kept, as a control: their normal
variation shows what an undisturbed task looks like over the same steps.

In [ ]:
import time

rows = []

def reading(step):
    """One full measurement. Both numbers, from the shared module."""
    forgotten = probe(hypernet, target, trainer, tasks[FORGET_TASK]["test"],
                      code_task=FORGET_TASK, classes=CLASSES, batch_size=EVAL_BATCH)
    kept = {t: probe(hypernet, target, trainer, tasks[t]["test"],
                     code_task=t, classes=CLASSES, batch_size=EVAL_BATCH)["accuracy"]
            for t in PROTECTED}
    pooled, per_tensor = relative_change(hypernet.weights_for(FORGET_TASK), reference,
                                         per_tensor=True)
    rows.append({
        "step": step,
        "removed_task_accuracy_pct": forgotten["accuracy"],
        "removed_task_distinct_classes": forgotten["distinct_classes"],
        "removed_task_chosen_class": forgotten["chosen_class"],
        "removed_task_class_counts": forgotten["class_counts"],
        "input_sensitivity": forgotten["input_sensitivity"],
        "rel_dW": pooled,
        "rel_dW_largest_tensor": max(per_tensor, key=per_tensor.get),
        "rel_dW_largest_value": max(per_tensor.values()),
        "weight_size": weight_size(hypernet, FORGET_TASK),
        "kept_mean_accuracy_pct": sum(kept.values()) / len(kept),
        "kept_accuracies_pct": kept,
    })
    return rows[-1]

began = time.perf_counter()
with Dashboard(f"Unlearning task {FORGET_TASK}: order {ORDER}, seed {SEED}") as board:
    # One tick per unlearning step, so the bar tracks the slow part honestly.
    bar = board.bar(config.burn_in, f"Unlearn task {FORGET_TASK}", unit="step")
    board.note(f"Reading before any unlearning, then every {INTERVAL} steps")
    reading(0)

    def on_step(record):
        """Called by the unlearning loop after each update."""
        step = record["step"]
        bar.update(1)
        if step > 0 and step % INTERVAL == 0:
            board.note(f"step {step}: measuring")
            row = reading(step)
            board.note(f"step {step}: accuracy {row['removed_task_accuracy_pct']:.1f}%, "
                       f"sensitivity {row['input_sensitivity']:.2e}")

    _forget_generated(trainer, FORGET_TASK, PROTECTED, config.burn_in, on_step)
    if rows[-1]["step"] != config.burn_in:
        reading(config.burn_in)
    bar.close()

ELAPSED_MIN = (time.perf_counter() - began) / 60
print(f"Finished {config.burn_in} unlearning steps and {len(rows)} readings "
      f"in {ELAPSED_MIN:.1f} minutes.")

## 9. The comparison

The table puts the two measurements beside each other as a fraction of where they started.

This is the point of the notebook. Read the `weight_size` column against the
`input_sensitivity` column. If the weights move modestly while sensitivity collapses, a
weight measurement on its own would not have found the problem.

In [ ]:
start, final = rows[0], rows[-1]

# Table 1's header is the widest line printed, so every rule, heading and
# banner is drawn to its width and nothing overhangs.
TABLE1_HEADER = (f"{'step':>5} {'accuracy':>9} {'classes':>8} | "
                 f"{'weight size':>12} {'of start':>9} {'rel_dW':>8} | "
                 f"{'sensitivity':>12} {'of start':>10}")
WIDTH = len(TABLE1_HEADER)

def rule(title):
    print(f"\n{title}\n" + "-" * WIDTH)

print("=" * WIDTH)
print(f"UNLEARNING TASK {FORGET_TASK}".center(WIDTH))
print(f"study {STUDY_ID} | order {ORDER} | seed {SEED} | "
      f"{config.burn_in} steps | {ELAPSED_MIN:.1f} min".center(WIDTH))
print("=" * WIDTH)

rule("TABLE 1. The two measurements, step by step")
print(TABLE1_HEADER)
print(f"{'':>5} {'percent':>9} {'used':>8} | {'':>12} {'percent':>9} {'ratio':>8} | "
      f"{'':>12} {'ratio':>10}")
print("-" * WIDTH)
for row in rows:
    print(f"{row['step']:>5} {row['removed_task_accuracy_pct']:>9.1f} "
          f"{row['removed_task_distinct_classes']:>5}/{CLASSES:<2} | "
          f"{row['weight_size']:>12.1f} "
          f"{100 * row['weight_size'] / start['weight_size']:>9.1f} "
          f"{row['rel_dW']:>8.4f} | "
          f"{row['input_sensitivity']:>12.3e} "
          f"{row['input_sensitivity'] / start['input_sensitivity']:>10.2e}")
print("-" * WIDTH)
print("accuracy    percent correct on the removed task's 500 validation images")
print("classes     how many of the ten classes the model ever answers. 1 means one")
print("            fixed answer for every image, which is what scores exactly 10.0")
print("weight size size of the generated weights before scaling")
print("rel_dW      ||W_now - W_start|| / ||W_start||, pooled over every tensor")
print("sensitivity widest range any single output score covers across the test set")

rule("TABLE 2. The tasks being kept, as a control")
kept_header = f"{'step':>5} " + " ".join(f"{('task ' + t):>8}" for t in PROTECTED) + f" {'mean':>8}"
print(kept_header)
print("-" * max(WIDTH, len(kept_header)))
for row in rows:
    cells = " ".join(f"{row['kept_accuracies_pct'][t]:>8.1f}" for t in PROTECTED)
    print(f"{row['step']:>5} {cells} {row['kept_mean_accuracy_pct']:>8.1f}")
print("-" * WIDTH)
print("All values are percent correct. These tasks were not targeted. Their ordinary")
print("movement over the same steps is what an undisturbed task looks like.")

rule("TABLE 3. Where the removed task's predictions went")
counts_header = f"{'step':>5} " + " ".join(f"{('c' + str(i)):>6}" for i in range(CLASSES))
print(counts_header)
print("-" * max(WIDTH, len(counts_header)))
for row in rows:
    print(f"{row['step']:>5} " + " ".join(f"{n:>6}" for n in row['removed_task_class_counts']))
print("-" * WIDTH)
print("Counts of the 500 validation images assigned to each class. The test set holds")
print("50 of each class, so one column of 500 gives exactly 50 correct, or 10.0 percent.")

weight_fraction = final["weight_size"] / start["weight_size"]
sensitivity_fraction = final["input_sensitivity"] / start["input_sensitivity"]

rule(f"SUMMARY. Over {final['step']} unlearning steps")
print(f"{'Weights moved by':<44}{final['rel_dW']:>12.1%} of their own size")
print(f"{'Weight size fell to':<44}{weight_fraction:>12.1%} of its starting value")
print(f"{'Input sensitivity fell to':<44}{sensitivity_fraction:>12.2e} of its starting value")
print(f"{'Removed task accuracy':<44}"
      f"{start['removed_task_accuracy_pct']:>7.1f}% to {final['removed_task_accuracy_pct']:.1f}%")
print(f"{'Removed task classes used':<44}"
      f"{start['removed_task_distinct_classes']:>8} to {final['removed_task_distinct_classes']}")
print(f"{'Kept tasks, mean accuracy':<44}"
      f"{start['kept_mean_accuracy_pct']:>7.1f}% to {final['kept_mean_accuracy_pct']:.1f}%")
print(f"{'Tensor that moved most':<44}{final['rel_dW_largest_tensor']:>12}")

rule("READING")
import math

# Compare the two on the same footing: how many factors of ten each moved.
# This is the honest comparison. An earlier version of this cell said the
# weights "moved a modest amount", which the measurements contradict: rel_dW
# reaches 0.63, a large move. The difference is not that one moved and the
# other did not. It is that they moved on scales that cannot be compared
# directly, and only one of them can reach zero.
sensitivity_decades = math.log10(start["input_sensitivity"] / final["input_sensitivity"])
print(f"{'Relative weight change':<30}{final['rel_dW']:>10.3f}")
print(f"{'Input sensitivity fell by':<30}{sensitivity_decades:>10.1f} factors of ten")
print()
print("On rel_dW, 0 means the weights are unchanged and 1 means they moved as far")
print("as their own size. Weights driven toward zero land near 1, so the scale has")
print("a ceiling. Input sensitivity has none.")
print()

if final["rel_dW"] < 1.0 and sensitivity_decades > 3:
    print("The two measurements disagree about how severe this is, which is why both")
    print("are needed.")
    print()
    print(f"  rel_dW reached {final['rel_dW']:.2f}. Read alone, that says the weights changed a")
    print("  lot, which is true, and it stops there. It is near the top of a scale")
    print("  that cannot go much further.")
    print()
    print(f"  Input sensitivity fell by {sensitivity_decades:.1f} factors of ten. With no ceiling it")
    print("  can show that the model did not merely change: it stopped responding to")
    print("  its input at all.")
    print()
    print("So a bounded weight measurement cannot separate a model that changed a")
    print("great deal from one that stopped working, because both sit near the top of")
    print("its range. Only the behavioural measurement tells them apart.")
elif sensitivity_decades <= 3:
    print("Input sensitivity did not collapse here, which is not what earlier runs")
    print("found. Record the numbers and say so plainly rather than explaining it away.")
else:
    print(f"rel_dW reached {final['rel_dW']:.2f}, at or past the weights' own size. Record both")
    print("numbers and describe what each one shows, without choosing between them.")

## 9b. Record which neurons fire, after unlearning

The same reading, on the same images, with the same control task. The model has now had
its unlearning request applied, so any difference comes from that.

Read the table from the top down. The stage where the response first collapses is where
the task stops. If the control column falls by the same amount, the damage is not
specific to the forgotten task.

In [ ]:
import statistics

profiles_after = {
    FORGET_TASK: activation_profile(FORGET_TASK),
    CONTROL_TASK: activation_profile(CONTROL_TASK),
}


def stage_of(layer):
    """Group the many ReLU readings into the five parts of a ResNet.

    Fifty rows is not readable. The question is where the signal dies, and the
    stage is a fine enough answer for that.
    """
    name = layer.split("#")[0]
    return name.split(".")[0] if "." in name else "stem"


def by_stage(profile):
    """Mean response per stage, averaged over every channel in it."""
    grouped = {}
    for layer, values in profile.items():
        grouped.setdefault(stage_of(layer), []).extend(values["mean"])
    return {stage: statistics.fmean(values) for stage, values in grouped.items()}


def silent_fraction(before, after):
    """Share of channels whose peak response fell below the threshold.

    Measured against each channel's own earlier peak, not an absolute number,
    because the whole network's scale moves during unlearning.
    """
    silent = total = 0
    for layer, values in after.items():
        for index, peak in enumerate(values["peak"]):
            was = before[layer]["peak"][index]
            total += 1
            if was > 0 and peak < SILENT_FRACTION * was:
                silent += 1
    return 100.0 * silent / total if total else float("nan")


WIDTH = 78
print("=" * WIDTH)
print(f"WHICH NEURONS STILL FIRE, order {ORDER}, seed {SEED}".center(WIDTH))
print(f"forgotten task {FORGET_TASK} | control task {CONTROL_TASK}".center(WIDTH))
print("=" * WIDTH)

forgotten_before, forgotten_after = by_stage(profiles_before[FORGET_TASK]), by_stage(profiles_after[FORGET_TASK])
control_before, control_after = by_stage(profiles_before[CONTROL_TASK]), by_stage(profiles_after[CONTROL_TASK])

print(f"\n{'stage':<10}{'forgotten before':>18}{'after':>12}{'kept x':>9}"
      f"{'control before':>16}{'after':>12}{'kept x':>9}")
print("-" * WIDTH)
for stage in sorted(forgotten_after):
    f0, f1 = forgotten_before[stage], forgotten_after[stage]
    c0, c1 = control_before[stage], control_after[stage]
    print(f"{stage:<10}{f0:>18.4f}{f1:>12.4f}{(f1 / f0 if f0 else float('nan')):>9.3f}"
          f"{c0:>16.4f}{c1:>12.4f}{(c1 / c0 if c0 else float('nan')):>9.3f}")
print("-" * WIDTH)
print("'kept x' is the share of the earlier response that remains. 1.000 means")
print("unchanged. A number near zero means that stage stopped responding.")

SILENT_FORGOTTEN = silent_fraction(profiles_before[FORGET_TASK], profiles_after[FORGET_TASK])
SILENT_CONTROL = silent_fraction(profiles_before[CONTROL_TASK], profiles_after[CONTROL_TASK])
print(f"\nChannels that went silent, below {SILENT_FRACTION:.0%} of their own earlier peak:")
print(f"  forgotten task {FORGET_TASK}: {SILENT_FORGOTTEN:.1f}% of {channels} channels")
print(f"  control task {CONTROL_TASK}:   {SILENT_CONTROL:.1f}% of {channels} channels")
print()
print("How to read this. A high figure for the forgotten task and a low one for the")
print("control means the request did what it was asked and nothing more. Two high")
print("figures mean the generator itself was damaged, not just that one task. Two low")
print("figures mean the task stopped working for some reason other than silent units,")
print("and the per-stage table above is then the thing to look at.")

## 10. Save

Three files in one timestamped folder: the full record as JSON, and the two tables above
as CSV so they open in a spreadsheet without any parsing.

Each file is read back after writing. `SAVE_VERIFIED` stays false if anything is missing,
and section 12 refuses to release the GPU while it is false, so a failed save cannot lose
the run.

In [ ]:
import csv, datetime, platform

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
EXPORT = CACHE / "diagnostics" / f"weight_change_and_sensitivity_{stamp}"
EXPORT.mkdir(parents=True, exist_ok=True)

record = {
    "notebook": "27_weight_change_and_sensitivity",
    "question": ("How far did the weights move, and does the model still respond to its "
                 "input? Measured together on one unlearning request."),
    "definitions": {
        "rel_dW": ("||W_now - W_start||_F / ||W_start||_F, pooled over every generated "
                   "tensor, float64. Same definition as rel_dW in "
                   "reference/adapter-plasticity-study."),
        "input_sensitivity": ("The widest range any single output score covers across the "
                              "validation set, measured in evaluation mode on raw scores."),
        "weight_size": "Norm of the generator's raw output for this task, before scaling.",
    },
    "code_revision": HEAD,
    "study_pinned_revision": STUDY_CODE_VERSION,
    "study_id": STUDY_ID, "order": ORDER, "seed": SEED,
    "forget_task": FORGET_TASK, "protected_tasks": PROTECTED,
    "interval": INTERVAL, "steps": config.burn_in,
    "classes_per_task": CLASSES, "config": plan["config"],
    "elapsed_min": ELAPSED_MIN,
    "environment": {
        "device": DEVICE,
        "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
        "torch": torch.__version__, "cuda": torch.version.cuda,
        "python": sys.version.split()[0], "platform": platform.platform(),
    },
    "readings": rows,
}
(EXPORT / "summary.json").write_text(json.dumps(record, indent=2) + "\n", encoding="utf-8")

# Table 1 and 2 as one wide CSV: one row per reading, one column per quantity.
with (EXPORT / "readings.csv").open("w", newline="", encoding="utf-8") as stream:
    columns = ["step", "removed_task_accuracy_pct", "removed_task_distinct_classes",
               "removed_task_chosen_class", "weight_size", "rel_dW",
               "rel_dW_largest_tensor", "rel_dW_largest_value", "input_sensitivity",
               "kept_mean_accuracy_pct"] + [f"kept_task_{t}_pct" for t in PROTECTED]
    writer = csv.DictWriter(stream, fieldnames=columns, extrasaction="ignore")
    writer.writeheader()
    for row in rows:
        line = dict(row)
        line.update({f"kept_task_{t}_pct": v for t, v in row["kept_accuracies_pct"].items()})
        writer.writerow(line)

# Table 3 separately, because it has one column per class rather than per quantity.
with (EXPORT / "class_counts.csv").open("w", newline="", encoding="utf-8") as stream:
    writer = csv.writer(stream)
    writer.writerow(["step"] + [f"class_{i}" for i in range(CLASSES)])
    for row in rows:
        writer.writerow([row["step"]] + row["removed_task_class_counts"])

# Read every file back. A file that cannot be reopened and parsed did not save.
SAVE_VERIFIED = True
for name in ("summary.json", "readings.csv", "class_counts.csv"):
    path = EXPORT / name
    try:
        text = path.read_text(encoding="utf-8")
        if name.endswith(".json"):
            json.loads(text)
        elif len(text.splitlines()) != len(rows) + 1:
            raise ValueError(f"expected {len(rows) + 1} lines, found {len(text.splitlines())}")
        print(f"  verified  {name:<20} {path.stat().st_size:>8,} bytes")
    except Exception as problem:
        SAVE_VERIFIED = False
        print(f"  FAILED    {name:<20} {problem}")

SAVED_LOCATION = str(EXPORT)
print(f"\nSaved locally: {EXPORT}")

if SAVE_TO_R2 and SAVE_VERIFIED:
    destination = PREFIX + "/diagnostics/" + EXPORT.name
    with Dashboard("Uploading results") as board:
        bar = board.bar(3, "Upload", unit="file")
        for name in ("summary.json", "readings.csv", "class_counts.csv"):
            store.upload(EXPORT / name, f"{destination}/{name}")
            bar.update(1)
        bar.close()
    SAVED_LOCATION = destination
    print("Uploaded to R2:", destination)
elif SAVE_TO_R2:
    print("Not uploading: the local files did not verify.")

## 11. Release the GPU

A finished notebook holding an A100 costs GPU time for nothing. This disconnects the Colab
runtime, but only once every file has been written and read back. If anything failed to
save it stays connected, so the run is not lost.

In [ ]:
if not SAVE_VERIFIED:
    print("NOT releasing the GPU: some output files are missing or unreadable.")
    print("Rerun section 10 before closing this runtime, or the measurements are lost.")
elif not RELEASE_GPU_WHEN_DONE:
    print("Results saved to:", SAVED_LOCATION)
    print("RELEASE_GPU_WHEN_DONE is off, so the runtime stays connected.")
elif not IN_COLAB:
    print("Results saved to:", SAVED_LOCATION)
    print("Not running in Colab, so there is no runtime to release.")
else:
    print("Results saved to:", SAVED_LOCATION)
    print("Share that path.")
    import gc
    # Drop the large objects first so the release does not wait on them.
    for name in ("hypernet", "target", "trainer", "state", "tasks", "reference"):
        globals().pop(name, None)
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    from google.colab import runtime
    print("Releasing the Colab runtime now.")
    runtime.unassign()

## 12. Using these measurements elsewhere

The two functions are now importable, so nothing needs copying into a new notebook:

```python
from uncle.measure import probe, relative_change, weight_size
```

`probe` returns accuracy, the per-class counts, how many classes the model ever chooses, and
input sensitivity. It switches to evaluation mode and restores the previous mode, which
matters: a BatchNorm layer in training mode rescales a nearly dead signal back to normal
size and hides the effect being measured.

`relative_change(current, reference)` is `rel_dW`. Pass `per_tensor=True` to also get the
same ratio for each tensor separately, which says where the change sits rather than only
how large it is.

For the adapter work that comes next, `relative_change` takes plain weight dictionaries, so
it applies unchanged to a LoRA adapter: pass the effective weights as `current` and the
frozen pretrained weights as `reference`.